# ==============================================================================
# Notebook 1: Carregamento e Pré-processamento dos Dados de Experimento
# ==============================================================================
# Objetivo:
# 1. Localizar e carregar os arquivos de log `interactions.jsonl` da estrutura
#    de diretórios dos experimentos.
# 2. Lidar com possíveis erros durante o carregamento (arquivos ausentes, JSON malformado).
# 3. Garantir que todas as colunas necessárias (`latency_ms`, `agent_trace`, etc.)
#    existam nos DataFrames, preenchendo com valores padrão se necessário.
# 4. Assegurar que os tipos de dados das colunas numéricas estejam corretos.
# 5. Salvar os dados agregados e pré-processados em um arquivo Pickle (`.pkl`)
#    para uso eficiente nos notebooks de análise subsequentes (02, 03, 04).
# ------------------------------------------------------------------------------

In [1]:
# --- Imports ---
import pandas as pd
import json
from pathlib import Path
from typing import Dict, List, Any
import pickle # Para salvar/carregar objetos Python (como nosso dicionário de DataFrames)
from IPython.display import display # Para visualização no notebook

In [2]:
# --- Configurações ---
# Diretório base onde os logs dos experimentos estão armazenados
# O código tentará encontrar a pasta correta automaticamente
BASE_LOG_DIR_SEARCH_PATHS = [
    Path('logs/experiments'),
    Path('../logs/experiments'),
    Path('../../logs/experiments'),
    Path('../../../logs/experiments'),
    Path('../../../../logs/experiments'),
    # Adicione outros caminhos se necessário
]

# Diretório para salvar os dados processados
OUTPUT_DATA_DIR = Path("processed_data")
OUTPUT_DATA_DIR.mkdir(exist_ok=True) # Cria o diretório se não existir

# Nome do arquivo para salvar os dados processados (formato Pickle)
PROCESSED_DATA_FILE = OUTPUT_DATA_DIR / "all_experiment_data.pkl"

print(f"Diretório de saída para dados processados: {OUTPUT_DATA_DIR.absolute()}")
print(f"Arquivo de saída: {PROCESSED_DATA_FILE.absolute()}")

Diretório de saída para dados processados: /Users/giossaurus/Developer/leia_tcc/notebooks/modelos_testes/processed_data
Arquivo de saída: /Users/giossaurus/Developer/leia_tcc/notebooks/modelos_testes/processed_data/all_experiment_data.pkl


In [3]:
# ==============================================================================
# Funções Auxiliares de Carregamento
# ==============================================================================

def _find_base_log_dir(search_paths: List[Path]) -> Path:
    """Tenta encontrar o diretório base de logs a partir de uma lista de caminhos."""
    for candidate in search_paths:
        resolved = candidate.resolve()
        if resolved.exists() and resolved.is_dir():
            print(f'\nBase de logs encontrada em: {resolved}')
            return resolved
    raise FileNotFoundError(
        'Não foi possível localizar a pasta de logs. Verifique as rotas de busca: '
        + ', '.join(str(path) for path in search_paths)
    )

def _load_interactions(interactions_file: Path) -> pd.DataFrame:
    """
    Lê um arquivo JSONL de interações e retorna um DataFrame.
    Inclui tratamento de erros por linha e garante colunas essenciais.
    """
    interactions: List[Dict[str, Any]] = []
    required_cols_defaults = {
        'user_input': '', 'agent_response': '', 'agent_trace': 'UNKNOWN',
        'latency_ms': 0.0, 'nlu_confidence': 0.0,
        'vram_used_mb': 0.0, 'ram_used_mb': 0.0,
        'model_name': 'UNKNOWN', 'scenario': 'UNKNOWN', 'experiment': 'UNKNOWN'
        # Adicione outras colunas que você espera ter, com seus padrões
    }
    numeric_cols = ['latency_ms', 'nlu_confidence', 'vram_used_mb', 'ram_used_mb']

    print(f"  Carregando arquivo: {interactions_file.name}...")
    try:
        with interactions_file.open('r', encoding='utf-8') as handler:
            for i, line in enumerate(handler):
                line = line.strip()
                if line:
                    try:
                        interaction_data = json.loads(line)
                        # Garante que todas as colunas requeridas existam
                        for col, default in required_cols_defaults.items():
                            if col not in interaction_data:
                                interaction_data[col] = default
                        interactions.append(interaction_data)
                    except json.JSONDecodeError as e:
                        print(f"  ⚠️ Erro ao decodificar linha {i+1} em {interactions_file.name}: {e} - Linha: {line[:100]}...")
                        # Adiciona um registro de erro
                        error_record = required_cols_defaults.copy()
                        error_record.update({"error": f"JSONDecodeError: {e}", "raw_line": line})
                        interactions.append(error_record)
    except Exception as e:
        print(f"  ❌ Erro GERAL ao ler o arquivo {interactions_file.name}: {e}")
        # Retorna um DataFrame vazio se não conseguir ler o arquivo
        return pd.DataFrame(columns=required_cols_defaults.keys())

    if not interactions:
        print(f"  ⚠️ Arquivo {interactions_file.name} está vazio ou não contém JSON válido.")
        return pd.DataFrame(columns=required_cols_defaults.keys())

    df = pd.DataFrame(interactions)

    # Garante a existência de todas as colunas requeridas após carregar todas as linhas
    for col, default in required_cols_defaults.items():
        if col not in df.columns:
            df[col] = default
            print(f"   INFO: Coluna '{col}' ausente globalmente em {interactions_file.name}, preenchida com '{default}'.")

    # Garante tipos corretos para colunas numéricas, tratando erros de conversão
    for col in numeric_cols:
        if col in df.columns:
             # errors='coerce' transforma o que não for número em NaN
             # fillna(0.0) substitui NaN por 0.0
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)
        else:
            # Se a coluna numérica ainda estiver faltando (caso raro), adiciona com 0.0
            df[col] = 0.0


    return df

In [6]:
#==============================================================================
# Função Principal de Carregamento e Agregação
# ==============================================================================

def load_and_process_experiment_data(base_log_dir: Path) -> Dict[str, pd.DataFrame]:
    """
    Vasculha a estrutura de logs, carrega, pré-processa e agrega os dados por modelo.

    Retorna um dicionário: { 'nome_do_modelo': DataFrame_com_todas_interacoes }
    """
    all_data: Dict[str, pd.DataFrame] = {}
    total_interactions_loaded = 0
    models_found = set()

    print("\nIniciando varredura e carregamento dos logs...")

    # Itera sobre experimentos (ex: phase4, phase4_quantized)
    for experiment_dir in sorted(base_log_dir.glob('*')):
        if not experiment_dir.is_dir():
            continue
        experiment_name = experiment_dir.name
        print(f"\nProcessando experimento: {experiment_name}")

        # Itera sobre modelos dentro do experimento
        for model_dir in sorted(experiment_dir.glob('*')):
            if not model_dir.is_dir():
                continue
            model_name = model_dir.name
            models_found.add(model_name)
            print(f" Modelo encontrado: {model_name}")

            aggregated_frames_for_model: List[pd.DataFrame] = []

            # Procura por subpastas de cenários (ex: standard_enem)
            found_scenarios = False
            for scenario_dir in sorted(model_dir.glob('*')):
                if scenario_dir.is_dir():
                    interactions_file = scenario_dir / 'interactions.jsonl'
                    if interactions_file.exists():
                        df = _load_interactions(interactions_file)
                        if not df.empty:
                            df['scenario'] = scenario_dir.name
                            df['experiment'] = experiment_name
                            df['model_name'] = model_name
                            aggregated_frames_for_model.append(df)
                            found_scenarios = True

            # Se não encontrou cenários, procura arquivo direto na pasta do modelo
            if not found_scenarios:
                 direct_interactions = model_dir / 'interactions.jsonl'
                 if direct_interactions.exists():
                     df = _load_interactions(direct_interactions)
                     if not df.empty:
                         df['scenario'] = 'default' # Cenário padrão
                         df['experiment'] = experiment_name
                         df['model_name'] = model_name
                         aggregated_frames_for_model.append(df)

            # Consolida os dados para este modelo (se houver algum)
            if aggregated_frames_for_model:
                model_df = pd.concat(aggregated_frames_for_model, ignore_index=True)
                # Adiciona ao dicionário geral, concatenando se o modelo já existia (de outro experimento)
                if model_name in all_data:
                    all_data[model_name] = pd.concat([all_data[model_name], model_df], ignore_index=True)
                else:
                    all_data[model_name] = model_df

                num_interactions = len(model_df)
                num_scenarios = model_df['scenario'].nunique()
                print(f"  ✓ Dados carregados para {model_name} neste experimento: {num_interactions} interações em {num_scenarios} cenários.")
                total_interactions_loaded += num_interactions
            else:
                 print(f"  ⚠️ Nenhum arquivo 'interactions.jsonl' válido encontrado para {model_name} neste experimento.")


    if not all_data:
        print('\n❌ Nenhum dado de interação foi carregado. Verifique a estrutura dos diretórios de log.')
    else:
        print(f'\n--- Resumo do Carregamento ---')
        print(f"  Modelos processados: {len(all_data)}")
        print(f"  Total de interações carregadas: {total_interactions_loaded}")
        for name, df_model in all_data.items():
            print(f"  - {name}: {len(df_model)} interações")

    return all_data

In [7]:
# ==============================================================================
# Execução Principal do Notebook
# ==============================================================================

processed_data: Dict[str, pd.DataFrame] = {}

try:
    # 1. Encontra o diretório base dos logs
    base_log_dir = _find_base_log_dir(BASE_LOG_DIR_SEARCH_PATHS)

    # 2. Carrega e pré-processa os dados
    processed_data = load_and_process_experiment_data(base_log_dir)

    # 3. Salva os dados processados em arquivo Pickle
    if processed_data:
        try:
            with open(PROCESSED_DATA_FILE, 'wb') as f:
                pickle.dump(processed_data, f)
            print(f"\n Dados processados salvos com sucesso em: {PROCESSED_DATA_FILE.absolute()}")

            # Mostra uma prévia dos dados carregados para um dos modelos
            if processed_data:
                example_model_name = list(processed_data.keys())[0]
                print(f"\n--- Prévia dos Dados Processados para '{example_model_name}' ---")
                display(processed_data[example_model_name].head())
                print("\n--- Informações do DataFrame (exemplo) ---")
                processed_data[example_model_name].info()


        except Exception as e:
            print(f"\n Erro ao salvar os dados processados em {PROCESSED_DATA_FILE}: {e}")
    else:
        print("\nNenhum dado foi processado, arquivo .pkl não foi salvo.")


except FileNotFoundError as e:
    print(f"\n ERRO FATAL: {e}")
    print("   Verifique se a pasta de logs existe e se os caminhos em BASE_LOG_DIR_SEARCH_PATHS estão corretos.")
except Exception as e:
    print(f"\n ERRO INESPERADO durante o carregamento/processamento: {e}")

print("\n--- Fim do Notebook 01 ---")

# Nota: O objeto `processed_data` (um dicionário onde as chaves são nomes de modelos
# e os valores são DataFrames com todas as interações daquele modelo) está agora
# salvo em 'processed_data/all_experiment_data.pkl'.
# Os próximos notebooks (02, 03, 04) deverão carregar este arquivo .pkl
# para iniciar suas respectivas análises.


Base de logs encontrada em: /Users/giossaurus/Developer/leia_tcc/logs/experiments

Iniciando varredura e carregamento dos logs...

Processando experimento: phase4_multimodel
 Modelo encontrado: Qwen_Qwen2.5-7B-Instruct
  Carregando arquivo: interactions.jsonl...
  Carregando arquivo: interactions.jsonl...
  Carregando arquivo: interactions.jsonl...
  Carregando arquivo: interactions.jsonl...
  Carregando arquivo: interactions.jsonl...
  ✓ Dados carregados para Qwen_Qwen2.5-7B-Instruct neste experimento: 29 interações em 5 cenários.
 Modelo encontrado: google_gemma-2-2b-it
  Carregando arquivo: interactions.jsonl...
  Carregando arquivo: interactions.jsonl...
  Carregando arquivo: interactions.jsonl...
  Carregando arquivo: interactions.jsonl...
  Carregando arquivo: interactions.jsonl...
  ✓ Dados carregados para google_gemma-2-2b-it neste experimento: 25 interações em 5 cenários.
 Modelo encontrado: mistralai_Mistral-7B-Instruct-v0.2
  Carregando arquivo: interactions.jsonl...
  Carr

,turn_number,timestamp,experiment_name,model_name,user_input,agent_response,session_id,condensed_question,nlu_label,nlu_confidence,agent_trace,response,latency_ms,chat_history,vram_used_mb,ram_used_mb,test_turn,scenario,experiment
0,1,2025-10-23T20:38:02.249538,phase4_multimodel,Qwen_Qwen2.5-7B-Instruct,(Enem/2015) A filosofia grega parece começar c...,Interessante pergunta! Me conta: o que você já...,phase4_multimodel_Qwen/Qwen2.5-7B-Instruct_202...,(Enem/2015) A filosofia grega parece começar c...,Análise de Exemplo,0.950000,EXECUTED_STANDARD (Análise de Exemplo),Interessante pergunta! Me conta: o que você já...,98292.35,"[{'role': 'user', 'content': '(Enem/2015) A fi...",0.0,21042.66,"{'expected_intent': 'Conceitual', 'test_focus'...",edge_cases_enem,phase4_multimodel
1,2,2025-10-23T20:41:28.381112,phase4_multimodel,Qwen_Qwen2.5-7B-Instruct,a) 3.\nb) 5.\nc) 6.\nd) 8.\ne) 10.\n\nQual est...,Interessante pergunta! Me conta: o que você já...,phase4_multimodel_Qwen/Qwen2.5-7B-Instruct_202...,"**\nDe acordo com Nietzsche, quais são as três...",Procedimental,0.847054,EXECUTED_STANDARD (Procedimental),Interessante pergunta! Me conta: o que você já...,206115.41,"[{'role': 'user', 'content': '(Enem/2015) A fi...",0.0,20493.72,"{'expected_intent': 'Conceitual', 'test_focus'...",edge_cases_enem,phase4_multimodel
2,3,2025-10-23T20:46:13.222413,phase4_multimodel,Qwen_Qwen2.5-7B-Instruct,(ENEM/2013) O CONTRIBUINTE QUE VENDE MAIS DE R...,**\n\nVamos lá! Você já sabe qual é a porcenta...,phase4_multimodel_Qwen/Qwen2.5-7B-Instruct_202...,(ENEM/2013) O CONTRIBUINTE QUE VENDE MAIS DE R...,Procedimental,0.950000,EXECUTED_STANDARD (Procedimental),**\n\nVamos lá! Você já sabe qual é a porcenta...,284831.42,"[{'role': 'user', 'content': '(Enem/2015) A fi...",0.0,20480.59,"{'expected_intent': 'Procedimental', 'test_foc...",edge_cases_enem,phase4_multimodel
3,4,2025-10-23T20:48:52.824548,phase4_multimodel,Qwen_Qwen2.5-7B-Instruct,1. (Enem/2015) A soda cáustica pode ser usada ...,**\n\nVamos começar com a primeira questão: qu...,phase4_multimodel_Qwen/Qwen2.5-7B-Instruct_202...,1. (Enem/2015) A soda cáustica pode ser usada ...,Conceitual,0.662319,EXECUTED_STANDARD (Conceitual),**\n\nVamos começar com a primeira questão: qu...,159586.25,"[{'role': 'user', 'content': 'a) 3. b) 5. c) 6...",0.0,20994.44,"{'expected_intent': 'Conceitual', 'test_focus'...",edge_cases_enem,phase4_multimodel
4,1,2025-10-23T21:05:49.433877,phase4_multimodel,Qwen_Qwen2.5-7B-Instruct,(Enem/2014) Uma criança deseja criar triângulo...,"**\n""Vamos focar no comprimento dos lados do t...",phase4_multimodel_Qwen/Qwen2.5-7B-Instruct_202...,(Enem/2014) Uma criança deseja criar triângulo...,Análise de Exemplo,0.950000,EXECUTED_STANDARD (Análise de Exemplo),"**\n""Vamos focar no comprimento dos lados do t...",21408.27,"[{'role': 'user', 'content': '(Enem/2014) Uma ...",0.0,22430.83,"{'expected_intent': 'Procedimental', 'test_foc...",guardrail_enem,phase4_multimodel



--- Informações do DataFrame (exemplo) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29 entries, 0 to 28
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   turn_number         29 non-null     int64  
 1   timestamp           29 non-null     object 
 2   experiment_name     29 non-null     object 
 3   model_name          29 non-null     object 
 4   user_input          29 non-null     object 
 5   agent_response      29 non-null     object 
 6   session_id          29 non-null     object 
 7   condensed_question  29 non-null     object 
 8   nlu_label           29 non-null     object 
 9   nlu_confidence      29 non-null     float64
 10  agent_trace         29 non-null     object 
 11  response            29 non-null     object 
 12  latency_ms          29 non-null     float64
 13  chat_history        29 non-null     object 
 14  vram_used_mb        29 non-null     float64
 15  ram_used_mb    